In [15]:
!pip install catboost

In [16]:
!pip install optuna

In [17]:
!pip install autogluon

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import StratifiedKFold
from autogluon.tabular import TabularPredictor

warnings.filterwarnings('ignore')

class CFG:
    seed = 42
    n_clusters = 20
    time_limit_base = 600   # 10 phút cho mô hình mồi
    time_limit_final = 1800 # 30 phút cho mô hình chốt hạ
    pseudo_threshold_low = 0.05  # Chắc chắn KHÔNG hỏng (< 5%)
    pseudo_threshold_high = 0.95 # Chắc chắn HỎNG (> 95%)
    data_root = "./dataset/public" if os.path.isdir("./dataset/public") else "./public"

def generate_spatial_target_encoding(train, test, target_col='underperforming'):
    print(" 🌍 Đang rèn vũ khí: OOF Spatial Target KNN (Khai thác tọa độ)...")

    # Chuẩn bị dữ liệu tọa độ
    train_coords = np.radians(train[['latitude', 'longitude']])
    test_coords = np.radians(test[['latitude', 'longitude']])
    y = train[target_col]

    # Khởi tạo cột mới
    train['geo_target_mean'] = 0.0
    test['geo_target_mean'] = 0.0

    # 1. Tính cho tập Train bằng Out-Of-Fold (OOF) để tránh Leakage
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG.seed)
    for tr_idx, val_idx in skf.split(train_coords, y):
        X_tr, y_tr = train_coords.iloc[tr_idx], y.iloc[tr_idx]
        X_val = train_coords.iloc[val_idx]

        # Dùng KNN để tìm 5 hàng xóm gần nhất và tính trung bình tỉ lệ lỗi
        knn = KNeighborsRegressor(n_neighbors=5, metric='haversine')
        knn.fit(X_tr, y_tr)
        train.loc[val_idx, 'geo_target_mean'] = knn.predict(X_val)

    # 2. Tính cho tập Test (Dùng toàn bộ tập Train)
    knn_full = KNeighborsRegressor(n_neighbors=5, metric='haversine')
    knn_full.fit(train_coords, y)
    test['geo_target_mean'] = knn_full.predict(test_coords)

    return train, test

def main():
    os.makedirs("./working", exist_ok=True)
    train_df = pd.read_csv(os.path.join(CFG.data_root, "train.csv"))
    test_df = pd.read_csv(os.path.join(CFG.data_root, "test.csv"))
    id_col, target_col = 'id', 'underperforming'

    # --- BƯỚC 1: TẠO ĐẶC TRƯNG SIÊU CẤP ---
    print(" 💡 Tạo đặc trưng mới...")
    train_df, test_df = generate_spatial_target_encoding(train_df, test_df, target_col)

    # Thêm đặc trưng đếm (Count Encoding) cho owner_bucket
    owner_counts = pd.concat([train_df['owner_bucket'], test_df['owner_bucket']]).value_counts()
    train_df['owner_count'] = train_df['owner_bucket'].map(owner_counts)
    test_df['owner_count'] = test_df['owner_bucket'].map(owner_counts)

    # Create squared features for plant_age and capacity_mw
    train_df['plant_age_sq'] = train_df['plant_age']**2
    test_df['plant_age_sq'] = test_df['plant_age']**2

    train_df['capacity_mw_sq'] = train_df['capacity_mw']**2
    test_df['capacity_mw_sq'] = test_df['capacity_mw']**2

    # Create an interaction feature
    train_df['age_x_capacity'] = train_df['plant_age'] * train_df['capacity_mw']
    test_df['age_x_capacity'] = test_df['plant_age'] * test_df['capacity_mw']

    # Calculate mean capacity_mw for each fuel_group
    fuel_group_capacity_mean = train_df.groupby('fuel_group')['capacity_mw'].mean()
    train_df['fuel_group_capacity_mean'] = train_df['fuel_group'].map(fuel_group_capacity_mean)
    test_df['fuel_group_capacity_mean'] = test_df['fuel_group'].map(fuel_group_capacity_mean)

    # Calculate mean plant_age for each fuel_group
    fuel_group_age_mean = train_df.groupby('fuel_group')['plant_age'].mean()
    train_df['fuel_group_age_mean'] = train_df['fuel_group'].map(fuel_group_age_mean)
    test_df['fuel_group_age_mean'] = test_df['fuel_group'].map(fuel_group_age_mean)

    # K-Means clustering for geographical groups
    print(" 🗺️ Tạo đặc trưng nhóm địa lý bằng K-Means...")
    all_coords = pd.concat([train_df[['latitude', 'longitude']], test_df[['latitude', 'longitude']]], axis=0).drop_duplicates().reset_index(drop=True)
    kmeans = KMeans(n_clusters=CFG.n_clusters, random_state=CFG.seed, n_init=10)
    kmeans.fit(all_coords)

    train_df['geo_cluster'] = kmeans.predict(train_df[['latitude', 'longitude']])
    test_df['geo_cluster'] = kmeans.predict(test_df[['latitude', 'longitude']])

    print("   -> Các đặc trưng mới đã được thêm vào train_df (sau K-Means):")
    print(train_df.head())

    # --- BƯỚC 2: TRAIN MÔ HÌNH MỒI (BASE MODEL) ---
    print("\n🔥 BƯỚC 1/2: Huấn luyện Mô hình mồi để lấy Nhãn Giả...")
    train_data_base = train_df.drop(columns=[id_col])
    test_data_base = test_df.drop(columns=[id_col])

    predictor_base = TabularPredictor(
        label=target_col, eval_metric='roc_auc', path='./working/ag_base'
    ).fit(train_data=train_data_base, presets='best_quality', time_limit=CFG.time_limit_base, verbosity=0)

    # --- BƯỚC 3: PSEUDO-LABELING (HACK DATA) ---
    print("\n💉 Đang trích xuất Nhãn Giả (Pseudo-labels) từ tập Test...")
    test_preds_base = predictor_base.predict_proba(test_data_base).iloc[:, 1]

    # Lọc ra những ca cực kỳ chắc chắn
    confident_idx = (test_preds_base < CFG.pseudo_threshold_low) | (test_preds_base > CFG.pseudo_threshold_high)
    pseudo_df = test_df[confident_idx].copy()

    # Gán nhãn cứng (0 hoặc 1) dựa trên xác suất
    pseudo_df[target_col] = (test_preds_base[confident_idx] > 0.5).astype(int)

    print(f"   -> Bắt sống được {len(pseudo_df)} mẫu Test tự tin! Nhét vào Train ngay...")

    # Trộn tập Train gốc và tập Pseudo (Test)
    train_df_extended = pd.concat([train_df, pseudo_df], axis=0).reset_index(drop=True)
    train_data_final = train_df_extended.drop(columns=[id_col])

    # --- BƯỚC 4: TRAIN MÔ HÌNH CHỐT HẠ (FINAL MODEL) ---
    print("\n☢️ BƯỚC 2/2: Huấn luyện Mô hình chốt hạ với tập dữ liệu đột biến...")
    predictor_final = TabularPredictor(
        label=target_col, eval_metric='roc_auc', path='./working/ag_final', verbosity=2
    ).fit(
        train_data=train_data_final,
        presets='best_quality',
        time_limit=CFG.time_limit_final,
        num_bag_folds=10,       # Ép chia 10 folds để cực kỳ vững vàng
        num_stack_levels=2      # Ép xếp chồng 3 lớp (Base -> L2 -> L3)
    )

    print("\n🏆 BẢNG XẾP HẠNG MÔ HÌNH CHỐT HẠ:")
    print(predictor_final.leaderboard())

    # --- BƯỚC 5: XUẤT FILE ---
    print("\n🔮 Đang lấy dự đoán cuối cùng...")
    final_preds = predictor_final.predict_proba(test_data_base).iloc[:, 1]

    submission = pd.DataFrame({id_col: test_df[id_col], target_col: final_preds})
    out_path = os.path.join("./working", "submission.csv")
    submission.to_csv(out_path, index=False)
    print(f"\n✅ Đã lưu vũ khí tối thượng tại: {out_path}")

if __name__ == "__main__":
    main()

   Power Plant Underperformance Prediction - AutoGluon V5
   Seed: 42 | Time Limit: 900s
 ⚡ Đang chế tạo đặc trưng: Phân cụm địa lý và Thống kê phân vị...


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       10.05 GB / 12.67 GB (79.3%)
Disk Space Avail:   74.40 GB / 107.72 GB (69.1%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` 


🚀 Kích hoạt AutoGluon: Chế độ 'best_quality'...


Leaderboard on holdout data (DyStack):
                     model  score_holdout  score_val eval_metric  pred_time_test  pred_time_val    fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0      WeightedEnsemble_L2       0.718835   0.757961     roc_auc        2.896075       1.703976  143.851499                 0.004591                0.004105           0.221139            2       True          6
1      WeightedEnsemble_L3       0.718574   0.758782     roc_auc        3.039770       1.860647  179.338722                 0.006327                0.003992           0.451569            3       True          9
2        LightGBMXT_BAG_L2       0.714694   0.750231     roc_auc        2.994066       1.852887  177.348675                 0.102582                0.153016          33.718315            2       True          7
3          LightGBM_BAG_L2       0.714381   0.750355     roc_auc        3.033442       1.856654  178.887153          


------------------------------
🏆 BẢNG XẾP HẠNG MÔ HÌNH
------------------------------
                      model  score_val eval_metric  pred_time_val    fit_time  \
0       WeightedEnsemble_L2   0.759745     roc_auc       3.830368  580.527303   
1           CatBoost_BAG_L1   0.742986     roc_auc       0.226185  191.173261   
2         LightGBMXT_BAG_L1   0.740393     roc_auc       0.624849   39.755413   
3   RandomForestEntr_BAG_L1   0.740131     roc_auc       0.679429    7.814205   
4   RandomForestGini_BAG_L1   0.738031     roc_auc       0.777170    5.141853   
5     ExtraTreesGini_BAG_L1   0.736240     roc_auc       0.478232    3.074222   
6           LightGBM_BAG_L1   0.736014     roc_auc       0.213859   38.939464   
7     ExtraTreesEntr_BAG_L1   0.734026     roc_auc       0.470279    2.679906   
8            XGBoost_BAG_L1   0.731901     roc_auc       0.241801   49.992093   
9    NeuralNetFastAI_BAG_L1   0.707531     roc_auc       0.441382  135.068426   
10    NeuralNetTorch_B